<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:

Astale Pillow version broke everything before any model loaded, and the fix required upgrading then restarting in exactly that sequence. Corporate CDNs block programmatic downloads, which is worth anticipating rather than discovering mid-session. A file uploading at 0.0 MB with no clear error message is harder to debug than any model issue. Checking inputs explicitly at each stage saves a lot of time. Keyword scanning belongs in Python, context should be trimmed, and caching is essential in a notebook where cells get re-run constantly.

In [4]:
!pip install -qU Pillow
print("✅ Pillow upgraded. Now go to Runtime → Restart session, then continue from Step 2.")

✅ Pillow upgraded. Now go to Runtime → Restart session, then continue from Step 2.


In [1]:
# Importing
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("❌ NO GPU DETECTED! Go to Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required.")

!pip install -qU transformers accelerate
!pip install -qU llama-index-core llama-index-readers-file llama-index-embeddings-huggingface pypdf
!pip install -qU pdf2image
!apt-get install -q poppler-utils

print("✅ Installation Complete.")

✅ GPU detected: Tesla T4 (15.6 GB VRAM)
Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Installation Complete.


In [2]:
# Load Embedding
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.


In [3]:
# Load Vision

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"⏳ Loading {VLM_MODEL_ID}...")

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"   # Explicitly 'cuda' — never 'auto'
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

model_device = next(vlm_model.parameters()).device
print(f"✅ VLM loaded on: {model_device}")
if str(model_device) == 'cpu':
    print("⚠️  WARNING: Model is on CPU. Each query will take 5-10 minutes!")

⏳ Loading Qwen/Qwen2.5-VL-3B-Instruct...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ VLM loaded on: cuda:0


In [4]:
# Configure LlamaIndex Settings

Settings.embed_model = embed_model
Settings.llm = None   # VQA handled by Qwen VLM directly
print("✅ LlamaIndex settings configured.")

LLM is explicitly disabled. Using MockLLM.
✅ LlamaIndex settings configured.


In [37]:
import os

PDF_FILE = "AstraZeneca-Q4-2025-earnings.pdf"

if not os.path.exists(PDF_FILE):
    raise FileNotFoundError(f"❌ Upload {PDF_FILE} to Colab first (drag into the Files panel on the left)")

print(f"✅ Found: {PDF_FILE} ({os.path.getsize(PDF_FILE)/1e6:.2f} MB)")

✅ Found: AstraZeneca-Q4-2025-earnings.pdf (1.55 MB)


In [40]:
PDF_FILE = "AstraZeneca-Q4-2025-earnings.pdf"
documents = SimpleDirectoryReader(input_files=[PDF_FILE]).load_data()
print(f"✅ Loaded {len(documents)} pages.")

✅ Loaded 39 pages.


In [41]:
import os
from llama_index.core import SimpleDirectoryReader

PDF_FILE = "AstraZeneca-Q4-2025-earnings.pdf"

# Check file exists
if not os.path.exists(PDF_FILE):
    print(f"❌ File not found. Files in /content/:")
    print(os.listdir('/content/'))
else:
    size_mb = os.path.getsize(PDF_FILE) / 1e6
    print(f"✅ File found — {size_mb:.1f} MB")
    documents = SimpleDirectoryReader(input_files=[PDF_FILE]).load_data()
    print(f"✅ Loaded {len(documents)} pages.")
    print(f"   Sample metadata: {documents[0].metadata}")

✅ File found — 1.5 MB
✅ Loaded 39 pages.
   Sample metadata: {'page_label': '1', 'file_name': 'AstraZeneca-Q4-2025-earnings.pdf', 'file_path': 'AstraZeneca-Q4-2025-earnings.pdf', 'file_type': 'application/pdf', 'file_size': 1545942, 'creation_date': '2026-03-23', 'last_modified_date': '2026-03-23'}


In [42]:
# Build the Vector Index
print("🧠 Building Vector Index...")
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=1)
print("✅ Index Ready!")

🧠 Building Vector Index...
✅ Index Ready!


In [43]:
# The Visual RAG Orchestrator

from pypdf import PdfReader, PdfWriter
from pdf2image import convert_from_path
from PIL import Image
import torch

def query_visual_rag(query_text, max_new_tokens=256):
    print(f"\n🔎 Querying LlamaIndex for: '{query_text}'...")

    model_device = next(vlm_model.parameters()).device
    if str(model_device) == 'cpu':
        print("⚠️  WARNING: VLM is on CPU. Change runtime to T4 GPU.")

    # 1. RETRIEVE
    nodes = retriever.retrieve(query_text)
    if not nodes:
        return "❌ No relevant information found in the index."

    # 2. LOCATE
    best_node = nodes[0]
    page_label = best_node.metadata.get('page_label')
    page_index = int(page_label) - 1 if page_label else 0
    print(f"📍 Found answer on Page {page_label} (Score: {best_node.score:.4f})")
    print(f"   Context Snippet: {best_node.text[:120]}...")

    # 3. RENDER
    print("🖼️  Rendering page as image...")
    pages = convert_from_path(
        PDF_FILE,
        first_page=page_index + 1,
        last_page=page_index + 1,
        dpi=150
    )
    page_image = pages[0]

    # 4. VISION — build Qwen2.5-VL prompt
    print(f"🚀 Sending page image to Qwen2.5-VL (max {max_new_tokens} tokens)...")
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": page_image},
            {"type": "text", "text": (
                f"You are an expert financial analyst reviewing Page {page_label} "
                "of the AstraZeneca FY and Q4 2025 Earnings Report. "
                "Answer the following question based on the text, tables, and "
                "charts visible on this page. If the answer involves a chart, "
                "describe the visual trend. If specific figures are requested, "
                "cite the exact numbers and state which table or section they appear in.\n\n"
                f"Question: {query_text}"
            )}
        ]
    }]

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt],
        images=[page_image],
        return_tensors="pt"
    ).to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,   # Required when do_sample=False
            top_p=None,
        )

    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    answer = vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]
    return answer

print("✅ query_visual_rag() function defined and ready.")

✅ query_visual_rag() function defined and ready.


In [44]:
# Revenue Table Extraction

from IPython.display import display, Markdown

q1 = ("What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, "
     "and how did each change compared to FY 2024?")
display(Markdown(f"### Task 1: {q1}"))
display(Markdown(query_visual_rag(q1)))

### Task 1: What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?


🔎 Querying LlamaIndex for: 'What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?'...
📍 Found answer on Page 1 (Score: 0.7171)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements  Glossary  
 
1 
 
 ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


According to the table provided in the image, AstraZeneca's total Product Sales for FY 2025 were $55,733 million, while their Alliance Revenue was $3,067 million. 

For Product Sales:
- The actual amount was $55,733 million.
- The CER (Change in Revenue) was 9%.
- The actual amount for FY 2024 was $55,733 million.
- The CER for FY 2024 was 9%.

For Alliance Revenue:
- The actual amount was $3,067 million.
- The CER was 39%.
- The actual amount for FY 2024 was $3,067 million.
- The CER for FY 2024 was 38%.

In [45]:
# Regional Revenue Breakdown

q2 = ("Which geographic region had the highest Total Revenue growth in FY 2025, "
     "and what was the growth rate at constant exchange rates?")
display(Markdown(f"### Task 2: {q2}"))
display(Markdown(query_visual_rag(q2)))

### Task 2: Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?


🔎 Querying LlamaIndex for: 'Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?'...
📍 Found answer on Page 34 (Score: 0.4615)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements  Glossary  
34 
 
No...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The geographic region with the highest Total Revenue growth in FY 2025, at constant exchange rates, was Emerging Markets, with a growth rate of 11%. This can be seen in Table 28 under the "Change" column for the "Emerging Markets" row.

In [46]:
# R&D Pipeline Interpretation

q3 = ("Which medicines received regulatory approvals in the US between November 2025 "
     "and February 2026, and for what indications?")
display(Markdown(f"### Task 3: {q3}"))
display(Markdown(query_visual_rag(q3)))

### Task 3: Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?


🔎 Querying LlamaIndex for: 'Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?'...
📍 Found answer on Page 10 (Score: 0.5477)
   Context Snippet: BioPharmaceuticals - CVRM 
Farxiga 
FY 2025 
$m 
Total  
Revenue  
% Change        
Actual        CER  
 
 Growth drive...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The information provided does not specify any medicines that received regulatory approvals in the US between November 2025 and February 2026. The document focuses on revenue drivers, R&D progress, sustainability, financial performance, and other relevant metrics but does not mention any specific approvals during that timeframe.

In [47]:
# Audit Mode Query

q4 = (
    "From the Cash Flow Statement (Table 12), provide the exact figures for: "
    "(1) Net Cash from Operating Activities, "
    "(2) Capital Expenditure, and "
    "(3) Free Cash Flow for FY 2025 and FY 2024. "
    "For every number you cite, state the TABLE NUMBER and PAGE NUMBER it comes from. "
    "Do not estimate — only report figures explicitly visible on the page."
)

display(Markdown(f"### Task 4 (Audit Mode): {q4}"))
answer_t4 = query_visual_rag(q4, max_new_tokens=400)
display(Markdown(answer_t4))

### Task 4 (Audit Mode): From the Cash Flow Statement (Table 12), provide the exact figures for: (1) Net Cash from Operating Activities, (2) Capital Expenditure, and (3) Free Cash Flow for FY 2025 and FY 2024. For every number you cite, state the TABLE NUMBER and PAGE NUMBER it comes from. Do not estimate — only report figures explicitly visible on the page.


🔎 Querying LlamaIndex for: 'From the Cash Flow Statement (Table 12), provide the exact figures for: (1) Net Cash from Operating Activities, (2) Capital Expenditure, and (3) Free Cash Flow for FY 2025 and FY 2024. For every number you cite, state the TABLE NUMBER and PAGE NUMBER it comes from. Do not estimate — only report figures explicitly visible on the page.'...
📍 Found answer on Page 20 (Score: 0.5755)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements  Glossary  
20 
 
Ca...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 400 tokens)...


To answer your question accurately, I'll need to extract the exact figures from the provided text and tables. Here's the information I can gather:

1. **Net Cash from Operating Activities**:
   - Table 12: Cash Flow summary: FY 2025
   - FY 2025: $14,575 million
   - FY 2024: $11,861 million

2. **Capital Expenditure**:
   - Table 12: Cash Flow summary: FY 2025
   - FY 2025: $3,270 million
   - FY 2024: $2,218 million

3. **Free Cash Flow**:
   - Table 12: Cash Flow summary: FY 2025
   - FY 2025: $7,767 million
   - FY 2024: $3,881 million

These figures are directly stated in the text and Tables 12 and 13 of the document.

In [48]:
# Verification Step

# Verification: surface the raw page text for manual cross-checking
audit_query = (
    "Cash Flow Statement Table 12 Net Cash Operating Activities "
    "Capital Expenditure Free Cash Flow"
)
nodes = retriever.retrieve(audit_query)
best = nodes[0]

print(f"Page retrieved: {best.metadata.get('page_label')}")
print(f"Similarity score: {best.score:.4f}")
print("\n--- Raw page text (for manual verification) ---")
print(best.text)

Page retrieved: 26
Similarity score: 0.5369

--- Raw page text (for manual verification) ---
Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements  Glossary  
26 
 
Table 20: Condensed consolidated statement of changes in equity 
 Share 
capital 
Share 
premium 
account 
Other 
reserves 
Retained 
earnings 
Total 
attributable 
to owners of 
the Parent 
Non-
controlling 
interests 
Total equity 
 $m  $m  $m  $m  $m  $m  $m 
At 1 Jan 2024 388  35,188  2,065  1,502  39,143  23  39,166  
Profit for the period -  -  -  7,035  7,035  6  7,041  
Other comprehensive expense   -  -  -  (799) (799) (1) (800) 
Transfer to Other reserves -  -  15  (15) -  -  -  
Transactions with owners        
Dividends -  -  -  (4,602) (4,602) -  (4,602) 
Dividends paid to non-controlling interests -  -  -  -  -  (4) (4) 
Issue of Ordinary Shares -  38  -  -  38  -  38  
Changes in non-controlling interests -  -  -  -  -  61  61  
Movement in shares held by Employee

In [49]:
# Helper: Render Any Page for Visual Inspection

from pdf2image import convert_from_path
from IPython.display import display as ipy_display

def show_page(page_number: int):
    """Render and display a specific page (1-indexed) from the AstraZeneca PDF."""
    pages = convert_from_path(
        PDF_FILE,
        first_page=page_number,
        last_page=page_number,
        dpi=120
    )
    ipy_display(pages[0])

# Example: show_page(12)  # Change the number to inspect any page
print("✅ show_page() helper defined. Call show_page(N) to inspect any page.")


✅ show_page() helper defined. Call show_page(N) to inspect any page.
